# 00 Setup and acquisition

Pulls both datasets into `data/raw` and loads them.

| Dataset | Kaggle ref | Version | Pulled |
|---|---|---|---|
| Steam | `fronkongames/steam-games-dataset` | v49 | 19 Sep 2026 |
| HowLongToBeat | `b4n4n4p0wer/how-long-to-beat-video-game-playtime-dataset` | v5 | 19 Sep 2026 |

The Steam dataset refreshes continuously. Do not re-download mid-project.

In [1]:
import urllib.request
import zipfile
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"

STEAM_ZIP = RAW / "steam_v49.zip"
HLTB_ZIP = RAW / "hltb_v5.zip"

In [2]:
# Download only if the pinned archive is missing
SOURCES = {
    STEAM_ZIP: "fronkongames/steam-games-dataset",
    HLTB_ZIP: "b4n4n4p0wer/how-long-to-beat-video-game-playtime-dataset",
}
for path, ref in SOURCES.items():
    if not path.exists():
        urllib.request.urlretrieve(f"https://www.kaggle.com/api/v1/datasets/download/{ref}", path)
    print(path.name, f"{path.stat().st_size / 1e6:.1f} MB")

steam_v49.zip 315.9 MB
hltb_v5.zip 18.1 MB


## Steam

The CSV header fuses `Discount` and `DLC count` into one name, giving 39 headers for 40 fields. Reading it as-is shifts every column one place, so the header is split before loading.

In [3]:
with zipfile.ZipFile(STEAM_ZIP) as z:
    with z.open("games.csv") as f:
        header = pd.read_csv(f, nrows=0).columns.tolist()

i = header.index("DiscountDLC count")
names = header[:i] + ["Discount", "DLC count"] + header[i + 1:]

with zipfile.ZipFile(STEAM_ZIP) as z:
    with z.open("games.csv") as f:
        steam = pd.read_csv(f, header=0, names=names, index_col=False)

steam.shape

(125855, 40)

In [4]:
steam.head()

,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,Discount,DLC count,About the game,...,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0,0.00,0,0,NaN,...,0,0,0,NaN,NaN,NaN,NaN,NaN,https://shared.akamai.steamstatic.com/store_it...,NaN
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,0,5.24,65,0,"Springtime, April: when the cherry trees come ...",...,0,8,0,minori,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure,"Adventure,Visual Novel,Anime,Cute",https://shared.akamai.steamstatic.com/store_it...,NaN
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,0,4.99,0,0,"Immerse yourself in the most beloved, mystical...",...,0,0,0,Somer Games,8floor,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec...",https://shared.akamai.steamstatic.com/store_it...,NaN
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0 - 20000,1,0,8.99,0,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",...,0,0,0,유진게임즈,유진게임즈,"Single-player,Steam Achievements,Family Sharing","Casual,Indie,Simulation",NaN,https://shared.akamai.steamstatic.com/store_it...,NaN
4,3631080,Maze Quest VR,"Apr 24, 2025",0 - 20000,0,0,4.99,0,0,Its not just a Maze; its a Quest! Enter the ca...,...,0,0,0,Reality Expanded LLC,Reality Expanded LLC,"Single-player,VR Only,Steam Leaderboards,Famil...","Action,Early Access",NaN,https://shared.akamai.steamstatic.com/store_it...,NaN


## HowLongToBeat

In [5]:
with zipfile.ZipFile(HLTB_ZIP) as z:
    with z.open("hltb_dataset_normalized.csv") as f:
        hltb = pd.read_csv(f)

hltb.shape

(166754, 28)

In [6]:
hltb.head()

,id,name,type,platform,genres,developer,publisher,release_date,release_precision,release_year,...,all_styles_polled,all_styles,single_player_polled,single_player,co_op_polled,co_op,versus_polled,versus,source_url,crawled_at
0,1,688(I) Hunter/Killer,game,PC,"Simulation, Strategy/Tactical",Sonalysts,Electronic Arts,1997-07-04,day,1997.0,...,15.0,15.93,8.0,10.88,NaN,NaN,NaN,NaN,https://howlongtobeat.com/game/1,2026-03-27T10:06:13Z
1,2,Beyond Good & Evil 2,game,NaN,"Action, Adventure",Ubisoft Montpellier,Ubisoft,0000-00-00,day,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://howlongtobeat.com/game/2,2026-03-27T10:06:14Z
2,18,'Splosion Man,game,Xbox 360,Action,Twisted Pixel Games,Microsoft Games Studios,2009-07-22,day,2009.0,...,62.0,8.76,32.0,7.80,1.0,9.0,NaN,NaN,https://howlongtobeat.com/game/18,2026-03-27T10:06:16Z
3,19,.hack//G.U. Vol. 1: Rebirth,game,"Nintendo Switch, PC, PlayStation 2, PlayStation 4","Action, Role-Playing",CyberConnect2,Bandai,2006-05-18,day,2006.0,...,181.0,23.41,76.0,18.52,NaN,NaN,NaN,NaN,https://howlongtobeat.com/game/19,2026-03-27T10:06:15Z
4,20,.hack//G.U. Vol. 2: Reminisce,game,"Nintendo Switch, PC, PlayStation 2, PlayStation 4","Action, Role-Playing",CyberConnect2,Bandai,2006-09-28,day,2006.0,...,88.0,26.62,41.0,19.82,NaN,NaN,NaN,NaN,https://howlongtobeat.com/game/20,2026-03-27T10:06:15Z
